In [ ]:
!pip install pytorch-tabnet

In [ ]:
# -----------------------------
# TabNet training without imblearn
# -----------------------------

import numpy as np
import pandas as pd
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix
from pytorch_tabnet.tab_model import TabNetClassifier
import torch

In [ ]:
# -----------------------------
# Settings
# -----------------------------
random_state = 42
test_size=0.2
device = "cuda" if torch.cuda.is_available() else "cpu"
print("Using device:", device)

Using device: cuda


In [ ]:
import pandas as pd

# Path to your processed clinical data CSV
clinical_data_path = "/kaggle/input/liver-clinical-processed/clinical_data_processed.csv"

# Load dataset
df = pd.read_csv(clinical_data_path)

# Target column
target_col = "Stage"

# Define features (X) and target (y)
X = df.drop(columns=[target_col])
y = df[target_col].astype(int)


In [ ]:
# Split data
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=test_size, stratify=y, random_state=random_state
)

In [ ]:
# -----------------------------
# Prepare data
# -----------------------------
# Assuming X_train, y_train, X_test, y_test are already defined as Pandas DataFrames/Series
X_train = X_train.copy()
y_train = y_train.copy()
X_test = X_test.copy()
y_test = y_test.copy()

# Encode labels if they are not numeric
if y_train.dtype != np.int64 and y_train.dtype != np.int32:
    le = LabelEncoder()
    y_train = le.fit_transform(y_train)
    y_test = le.transform(y_test)

In [ ]:
# Identify categorical columns by dtype (optional)
cat_columns = X_train.select_dtypes(include=['object', 'category']).columns.tolist()
cat_idxs = [X_train.columns.get_loc(col) for col in cat_columns]

# Standardize continuous features (optional, TabNet can handle unscaled features)
num_cols = [c for c in X_train.columns if c not in cat_columns]
if len(num_cols) > 0:
    scaler = StandardScaler()
    X_train[num_cols] = scaler.fit_transform(X_train[num_cols])
    X_test[num_cols] = scaler.transform(X_test[num_cols])

In [ ]:
# -----------------------------
# Manual oversampling of the tiny class
# -----------------------------
class_counts = pd.Series(y_train).value_counts()
min_class = class_counts.idxmin()
n_min_class = class_counts.min()
target_count = class_counts.sort_values().iloc[1]  # oversample to match second smallest class

X_min = X_train[y_train == min_class]
y_min = y_train[y_train == min_class]

n_repeat = target_count - n_min_class
X_train_res = pd.concat([X_train, X_min.sample(n_repeat, replace=True)], axis=0)
y_train_res = pd.concat([y_train, y_min.sample(n_repeat, replace=True)], axis=0)

print("Training set size before:", len(X_train), ", after oversampling tiny class:", len(X_train_res))

Training set size before: 329 , after oversampling tiny class: 385


In [ ]:
# -----------------------------
# Convert to NumPy for TabNet
# -----------------------------
X_train_np = X_train_res.values
y_train_np = y_train_res.values
X_test_np = X_test.values
y_test_np = y_test.values

In [ ]:
# -----------------------------
# TabNet Classifier
# -----------------------------
clf = TabNetClassifier(
    n_d=32, n_a=32, n_steps=5,
    gamma=1.5, n_independent=2, n_shared=2,
    cat_idxs=cat_idxs,
    cat_dims=[X_train[col].nunique() for col in cat_columns],
    cat_emb_dim=1,
    optimizer_fn=torch.optim.Adam,
    optimizer_params=dict(lr=2e-2),
    scheduler_params={"step_size":50, "gamma":0.9},
    scheduler_fn=torch.optim.lr_scheduler.StepLR,
    mask_type='entmax',
    verbose=10,
    seed=random_state,
    device_name=device
)

/usr/local/lib/python3.11/dist-packages/pytorch_tabnet/abstract_model.py:82: UserWarning: Device used : cuda
  warnings.warn(f"Device used : {self.device}")


In [ ]:
# -----------------------------
# Fit model
# -----------------------------
clf.fit(
    X_train_np, y_train_np,
    eval_set=[(X_test_np, y_test_np)],
    eval_name=['test'],
    eval_metric=['accuracy'],  # 'accuracy' or 'balanced_accuracy'
    max_epochs=200,
    patience=50,
    batch_size=32,
    virtual_batch_size=16
)

epoch 0  | loss: 2.2945  | test_accuracy: 0.3253  |  0:00:01s
epoch 10 | loss: 1.16051 | test_accuracy: 0.43373 |  0:00:05s
epoch 20 | loss: 1.14668 | test_accuracy: 0.44578 |  0:00:09s
epoch 30 | loss: 1.07335 | test_accuracy: 0.43373 |  0:00:13s
epoch 40 | loss: 1.05623 | test_accuracy: 0.3494  |  0:00:17s
epoch 50 | loss: 0.95784 | test_accuracy: 0.39759 |  0:00:22s
epoch 60 | loss: 0.94994 | test_accuracy: 0.37349 |  0:00:26s

Early stopping occurred at epoch 64 with best_epoch = 14 and best_test_accuracy = 0.51807


/usr/local/lib/python3.11/dist-packages/pytorch_tabnet/callbacks.py:172: UserWarning: Best weights from best epoch are automatically used!
  warnings.warn(wrn_msg)


In [ ]:
# -----------------------------
# Evaluate
# -----------------------------
y_pred = clf.predict(X_test_np)
acc = accuracy_score(y_test_np, y_pred)
print(f"\nTest accuracy: {acc:.4f}")

print("\nClassification report:")
print(classification_report(y_test_np, y_pred, digits=4))

cm = confusion_matrix(y_test_np, y_pred)
print("\nConfusion matrix:\n", cm)


Test accuracy: 0.5181

Classification report:
              precision    recall  f1-score   support

           1     0.1333    0.5000    0.2105         4
           2     0.6667    0.2105    0.3200        19
           3     0.5556    0.4839    0.5172        31
           4     0.6286    0.7586    0.6875        29

    accuracy                         0.5181        83
   macro avg     0.4960    0.4883    0.4338        83
weighted avg     0.5862    0.5181    0.5168        83


Confusion matrix:
 [[ 2  1  1  0]
 [ 7  4  5  3]
 [ 5  1 15 10]
 [ 1  0  6 22]]
